In [6]:
using DataFrames
using CSV
using Plots
using Dates
using Statistics

In [2]:
rat_df= CSV.read("../data/processed_rat_data.csv.gz", DataFrame)
rat_df = rat_df[rat_df.daily .== "24 hr", :] # only keep 24 hour data
trial_datetimes = DateTime.(rat_df.trial_datetime, dateformat"yyyy-mm-dd HH:MM:SS")

times = Time.(trial_datetimes)

# offset by 7.5 hours
offset = Hour(7) + Minute(30)
refdate = Date(2000, 1, 1)

wrapped_times = Time.(
    DateTime.(refdate, times) .- offset
)

time_hours = hour.(wrapped_times) .+
             minute.(wrapped_times) ./ 60 .+
             second.(wrapped_times) ./ 3600

592261-element Vector{Float64}:
  8.808611111111112
  9.212222222222222
  9.216388888888888
  9.223055555555556
  9.224722222222223
  9.344444444444445
  9.858888888888888
  9.862222222222222
  9.864166666666666
  9.868333333333334
  ⋮
 15.6075
 15.612777777777778
 15.640833333333333
 15.642777777777777
 15.657777777777778
 15.685277777777777
 15.709722222222222
 15.71361111111111
 15.722777777777777

In [18]:
histogram(
    time_hours;
    fontfamily = "helvetica",
    bins = 48,
    xlabel = "Time from Light On (H)",
    ylabel = "Count (thousands)",
    xlims = (0, 24),
    legend = false,
)

savefig("rat_trial_time_histogram.svg")

"c:\\Users\\senne\\Documents\\GitHub\\24_Hour_Behavior\\notebooks\\rat_trial_time_histogram.svg"

In [16]:
df = deepcopy(rat_df)
df.trial_datetime = DateTime.(df.trial_datetime, dateformat"y-m-d H:M:S")
df.session_date = Date.(df.trial_datetime)

df.correct = Float64.(df.correct) 
df.rt      = Float64.(df.rt)

daily = combine(groupby(df, [:name, :session_date]),
    nrow => :trials,
    :correct => mean => :acc,
    :rt      => mean => :rt_mean,
)

daily = combine(groupby(df, [:name, :session_date]),
    nrow => :trials,
    :correct => mean => :acc,
    :rt      => mean => :rt_mean,
)

days_per_rat = combine(groupby(daily, :name),
    nrow => :ndays
)

valid_rats = days_per_rat.name[days_per_rat.ndays .> 80]
daily = daily[in.(daily.name, Ref(valid_rats)), :]

daily.day = similar(daily.session_date, Int)
for sub in groupby(daily, :name)
    idxs = parentindices(sub)[1]
    daily.day[idxs] = collect(1:length(idxs))
end

mean_by_day = combine(groupby(daily, :day),
    :trials  => mean => :trials_mean,
    :acc     => mean => :acc_mean,
    :rt_mean => mean => :rt_mean_mean,
)

sort!(mean_by_day, :day)

function plot_metric(metric::Symbol, meanmetric::Symbol; ylabel="", title="")
    p = plot(legend=false, xlabel="Day", fontfamily="helvetica", ylabel=ylabel, title=title)

    # individual rats in light grey
    for sub in groupby(daily, :name)
        plot!(p, sub.day, sub[!, metric], color=:lightgrey, lw=1, alpha=0.9)

        if metric == :acc
            ylims!(p, 0.5, 1)
        end

    end

    # overall mean on top (default color)
    plot!(p, mean_by_day.day, mean_by_day[!, meanmetric], lw=3)
    return p
end


p_acc    = plot_metric(:acc,     :acc_mean;       ylabel="Accuracy")
# p_rt     = plot_metric(:rt_mean, :rt_mean_mean;  ylabel="RT (mean)",               title="Daily reaction time")
# p_trials = plot_metric(:trials,  :trials_mean;   ylabel="Trials/day",              title="Daily trials")

# # plot(p_acc, p_rt, p_trials, layout=(3,1), size=(800, 1100))
savefig("rat_daily_acc.svg")

"c:\\Users\\senne\\Documents\\GitHub\\24_Hour_Behavior\\notebooks\\rat_daily_acc.svg"